In [64]:
# Optimize the code
# pagination
# Store the data for each page
# connect to SQLight
# define ID or get specific ID (reference number) for jobs, so the common one will be detected.
# Check if I did everrything with the first plan


In [8]:
import time
import re
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup


# JobPosting class to store job information
class JobPosting:
    def __init__(self, title, location, salary, date_posted, closing_date, contract_type, working_pattern, job_link):
        self.reference_number = None
        self.title = title
        self.location = location
        self.working_pattern = working_pattern
        self.salary = salary
        self.date_posted = date_posted
        self.closing_date = closing_date
        self.contract_type = contract_type
        self.grade = None
        self.duration = None
        self.job_link = job_link
        self.job_summary = None
        self.main_duties = None
        self.job_description = None
        self.team_structure = None
        self.qualifications = None
        self.employer_name = None
        self.employer_contact = None
        self.employer_address = None
        self.disclosure_check = None
        self.certificate_of_sponsorship = None
        self.uk_registration = None
        self.pay_scheme = None

    def __repr__(self):
        return (
            f"Reference Number: {self.reference_number}\n\n"
            f"Job Title: {self.title}\n\n"
            f"Location: {self.location}\n\n"
            f"Working Pattern: {self.working_pattern}\n\n"
            f"Salary: {self.salary}\n\n"
            f"Date Posted: {self.date_posted}\n\n"
            f"Closing Date: {self.closing_date}\n\n"
            f"Contract Type: {self.contract_type}\n\n"
            f"Grade: {self.grade}\n\n"
            f"Duration: {self.duration}\n\n"
            f"Job Link: {self.job_link}\n\n"
            f"Job Summary: {self.job_summary}\n\n"
            f"Main Duties: {self.main_duties}\n\n"
            f"Job Description: {self.job_description}\n\n"
            f"Team Structure: {self.team_structure}\n\n"
            f"Qualifications: {self.qualifications}\n\n"
            f"Employer Name: {self.employer_name}\n\n"
            f"Employer Contact: {self.employer_contact}\n\n"
            f"Employer Address: {self.employer_address}\n\n"
            f"Disclosure Check: {self.disclosure_check}\n\n"
            f"Certificate of Sponsorship: {self.certificate_of_sponsorship}\n\n"
            f"UK Registration: {self.uk_registration}\n\n"
            f"Pay Scheme: {self.pay_scheme}\n\n"
        )


# Helper functions
def clean_text(text):
    return ' '.join(text.split()).strip() if text else 'N/A'

def clean_list(text):
    if not text:
        return 'N/A'
    text = re.sub(r'\b(?:e\.g|i\.e|etc|Dr|Mr|Mrs|Ms|Jr|Sr|vs|Inc|Ltd|Co|Prof|PhD|M\.D|B\.Sc)\.', 
                  lambda m: m.group(0).replace('.', '[DOT]'), text)
    return '\n'.join(f"- {sentence.strip()}." for sentence in text.split('.') if sentence.strip()).replace('[DOT]', '.')

def clean_text_with_strong_tags(text, tag):
    soup = BeautifulSoup(text, 'html.parser')
    for strong_tag in soup.find_all(tag):
        strong_tag.string = f"**{strong_tag.text}**"
    return ' '.join(soup.stripped_strings)


def extract_tag_text(soup, tag_name, partial_string, next_tag_stop='h2', nested_tag='p'):
    tag = soup.find(tag_name, string=lambda text: text and partial_string.lower() in text.lower())
    if tag:
        content = [clean_text_with_strong_tags(str(sibling), 'strong') for sibling in tag.find_next_siblings()
                   if sibling.name == nested_tag and sibling.name != next_tag_stop]
        return clean_text(' '.join(content)) if content else 'N/A'
    return 'N/A'


def extract_qualifications(soup, tag_name, partial_string, next_tag_stop='h2'):
    tag = soup.find(tag_name, string=lambda text: text and partial_string.lower() in text.lower())
    if tag:
        qualifications = [clean_text(li.text) for sibling in tag.find_next_siblings()
                          if sibling.name == 'ul' for li in sibling.find_all('li') if sibling.name != next_tag_stop]
        return ' '.join(qualifications) if qualifications else 'N/A'
    return 'N/A'


# Updated async function to extract detailed job information
async def extract_job_details(page, job_posting):
    try:
        await page.goto(job_posting.job_link, wait_until='networkidle')
        await page.wait_for_selector('main.nhsuk-main-wrapper')
        job_detail_html = await page.content()
        detail_soup = BeautifulSoup(job_detail_html, 'html.parser')
        main_content = detail_soup.find('main', class_='nhsuk-main-wrapper')
        
        if main_content:
            # Apply clean_list to the required fields
            job_posting.job_summary = clean_list(extract_tag_text(main_content, 'h3', 'summary'))
            job_posting.main_duties = clean_list(extract_tag_text(main_content, 'h3', 'duties'))
            job_posting.team_structure = clean_list(extract_tag_text(main_content, 'h3', 'about us'))
            job_posting.qualifications = clean_list(extract_qualifications(main_content, 'h2', 'specification'))
            job_posting.job_description = clean_list(extract_tag_text(main_content, 'h2', 'description'))
    
            # Other fields can remain as they were
            job_posting.working_pattern = extract_tag_text(main_content, 'h3', 'working pattern')
            employer_name_tag = main_content.find('p', id='employer_name_details')
            job_posting.employer_name = clean_text(employer_name_tag.text) if employer_name_tag else 'N/A'
    
            # Extract employer address
            address_fields = ['employer_address_line_1_a', 'employer_address_line_2_b', 'employer_town_c', 'employer_postcode_e']
            job_posting.employer_address = clean_text(' '.join([clean_text(main_content.find('p', id=field).text or '')
                                                                for field in address_fields if main_content.find('p', id=field)])) or 'N/A'
    
            employer_contact_tag = main_content.find('p', id='employer_website_url')
            if employer_contact_tag:
                employer_contact_link = employer_contact_tag.find('a', id='employer_website_url_link')
                job_posting.employer_contact = clean_text(employer_contact_link['href']) if employer_contact_link else 'N/A'
            else:
                job_posting.employer_contact = 'N/A'
    
            job_posting.disclosure_check = clean_text(main_content.find('div', id='dbs-container').text if main_content.find('div', id='dbs-container') else 'N/A')
            job_posting.certificate_of_sponsorship = clean_text(main_content.find('h3', id='tier-two-sponsorship').find_next('p').text if main_content.find('h3', id='tier-two-sponsorship') else 'N/A')
            job_posting.uk_registration = clean_text(main_content.find('h3', id='uk-registration').find_next('p').text if main_content.find('h3', id='uk-registration') else 'N/A')
            job_posting.pay_scheme = clean_text(main_content.find('p', id='payscheme-type').text if main_content.find('p', id='payscheme-type') else 'None')
            job_posting.grade = clean_text(main_content.find('p', id='payscheme-band').text if main_content.find('p', id='payscheme-band') else 'None')
            job_posting.reference_number = clean_text(main_content.find('p', id='trac-job-reference').text if main_content.find('p', id='trac-job-reference') else 'None')
            job_posting.duration = clean_text(main_content.find('p', id='contract_duration').text if main_content.find('p', id='contract_duration') else 'None')

    except Exception as e:
        print(f"Error extracting details for {job_posting.title}: {e}")
        
    
async def scrape_jobs_playwright(url):
    job_listings = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        page_number = 1
        
        try:
            while True:
                await page.goto(url)
                await page.wait_for_selector('.nhsuk-list.search-results')
                soup = BeautifulSoup(await page.content(), 'html.parser')
                job_elements = soup.find_all('li', class_='nhsuk-list-panel')

                for job in job_elements:
                    title_tag = job.find('a', {'data-test': 'search-result-job-title'})
                    title = clean_text(title_tag.text) if title_tag else 'N/A'
                    job_link = f"https://www.jobs.nhs.uk{title_tag['href']}" if title_tag else 'N/A'
                    location = clean_text(job.find('div', {'data-test': 'search-result-location'}).text or 'N/A')
                    salary = clean_text(job.find('li', {'data-test': 'search-result-salary'}).find('strong').text or 'N/A')
                    date_posted = clean_text(job.find('li', {'data-test': 'search-result-publicationDate'}).find('strong').text or 'N/A')
                    closing_date = clean_text(job.find('li', {'data-test': 'search-result-closingDate'}).find('strong').text or 'N/A')
                    contract_type = clean_text(job.find('li', {'data-test': 'search-result-jobType'}).find('strong').text or 'N/A')
                    working_pattern = clean_text(job.find('li', {'data-test': 'search-result-workingPattern'}).find('strong').text or 'N/A')

                    job_posting = JobPosting(title, location, salary, date_posted, closing_date, contract_type, working_pattern, job_link)
                    await extract_job_details(page, job_posting)
                    job_listings.append(job_posting)

                # Print message after scraping the page
                print(f"Page {page_number} scraped")

                # Check for pagination
                next_page_tag = soup.find('li', class_='nhsuk-pagination-item--next')
                if next_page_tag:
                    next_page_link = next_page_tag.find('a', {'data-test': 'search-next-page'})
                    if next_page_link and 'href' in next_page_link.attrs:
                        url = f"https://www.jobs.nhs.uk{next_page_link['href']}"
                        page_number += 1
                    else:
                        break  # No more pages
                else:
                    break  # No more next page

        except Exception as e:
            print(f"Error while scraping page {page_number}: {e}")
        finally:
            await browser.close()

    return job_listings

if __name__ == "__main__":
    base_url = "https://www.jobs.nhs.uk/candidate/search/results?"
    keyword = "FY2, CT1, CT2, ST1, ST2, ST3, LAS, Trust doctor, Trust grade"
    pay_band = "SPECIALTY_DOCTOR,FOUNDATION_DOCTOR,DOCTOR_OTHER"
    pay_range = "30-40,40-50"
    sort_by = "publicationDateDesc"
    language = "en"

    url = (f"{base_url}keyword={keyword.replace(' ', '%20')}&payBand={pay_band}&payRange={pay_range}"
           f"&skipPhraseSuggester=true&searchFormType=sortBy&sort={sort_by}&language={language}")

    start_time = time.time()
    jobs = await scrape_jobs_playwright(url)  # Use await here directly in environments with an active event loop
    print(f"Scraping completed in {time.time() - start_time:.2f} seconds")

    for job in jobs:
        print(job)
        print("-" * 50)



Page 1 scraped


CancelledError: 

In [2]:
import time
import re
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup


# JobPosting class to store job information
class JobPosting:
    def __init__(self, title, location, salary, date_posted, closing_date, contract_type, working_pattern, job_link):
        self.reference_number = None
        self.title = title
        self.location = location
        self.working_pattern = working_pattern
        self.salary = salary
        self.date_posted = date_posted
        self.closing_date = closing_date
        self.contract_type = contract_type
        self.grade = None
        self.duration = None
        self.job_link = job_link
        self.job_summary = None
        self.main_duties = None
        self.job_description = None
        self.team_structure = None
        self.qualifications = None
        self.employer_name = None
        self.employer_contact = None
        self.employer_address = None
        self.disclosure_check = None
        self.certificate_of_sponsorship = None
        self.uk_registration = None
        self.pay_scheme = None

    def __repr__(self):
        return (
            f"Reference Number: {self.reference_number}\n\n"
            f"Job Title: {self.title}\n\n"
            f"Location: {self.location}\n\n"
            f"Working Pattern: {self.working_pattern}\n\n"
            f"Salary: {self.salary}\n\n"
            f"Date Posted: {self.date_posted}\n\n"
            f"Closing Date: {self.closing_date}\n\n"
            f"Contract Type: {self.contract_type}\n\n"
            f"Grade: {self.grade}\n\n"
            f"Duration: {self.duration}\n\n"
            f"Job Link: {self.job_link}\n\n"
            f"Job Summary: {self.job_summary}\n\n"
            f"Main Duties: {self.main_duties}\n\n"
            f"Job Description: {self.job_description}\n\n"
            f"Team Structure: {self.team_structure}\n\n"
            f"Qualifications: {self.qualifications}\n\n"
            f"Employer Name: {self.employer_name}\n\n"
            f"Employer Contact: {self.employer_contact}\n\n"
            f"Employer Address: {self.employer_address}\n\n"
            f"Disclosure Check: {self.disclosure_check}\n\n"
            f"Certificate of Sponsorship: {self.certificate_of_sponsorship}\n\n"
            f"UK Registration: {self.uk_registration}\n\n"
            f"Pay Scheme: {self.pay_scheme}\n\n"
        )


# Helper functions
def clean_text(text):
    return ' '.join(text.split()).strip() if text else 'N/A'

def clean_list(text):
    if not text:
        return 'N/A'
    text = re.sub(r'\b(?:e\.g|i\.e|etc|Dr|Mr|Mrs|Ms|Jr|Sr|vs|Inc|Ltd|Co|Prof|PhD|M\.D|B\.Sc)\.', 
                  lambda m: m.group(0).replace('.', '[DOT]'), text)
    return '\n'.join(f"- {sentence.strip()}." for sentence in text.split('.') if sentence.strip()).replace('[DOT]', '.')

def clean_text_with_strong_tags(text, tag):
    soup = BeautifulSoup(text, 'html.parser')
    for strong_tag in soup.find_all(tag):
        strong_tag.string = f"**{strong_tag.text}**"
    return ' '.join(soup.stripped_strings)

def extract_tag_text(soup, tag_name, partial_string, next_tag_stop='h2', nested_tag='p'):
    tag = soup.find(tag_name, string=lambda text: text and partial_string.lower() in text.lower())
    if tag:
        content = [clean_text_with_strong_tags(str(sibling), 'strong') for sibling in tag.find_next_siblings()
                   if sibling.name == nested_tag and sibling.name != next_tag_stop]
        return clean_text(' '.join(content)) if content else 'N/A'
    return 'N/A'

def extract_qualifications(soup, tag_name, partial_string, next_tag_stop='h2'):
    tag = soup.find(tag_name, string=lambda text: text and partial_string.lower() in text.lower())
    if tag:
        qualifications = [clean_text(li.text) for sibling in tag.find_next_siblings()
                          if sibling.name == 'ul' for li in sibling.find_all('li') if sibling.name != next_tag_stop]
        return ' '.join(qualifications) if qualifications else 'N/A'
    return 'N/A'


# Updated async function to extract detailed job information
async def extract_job_details(page, job_posting):
    try:
        await page.goto(job_posting.job_link, wait_until='networkidle')
        await page.wait_for_selector('main.nhsuk-main-wrapper')
        job_detail_html = await page.content()
        detail_soup = BeautifulSoup(job_detail_html, 'html.parser')
        main_content = detail_soup.find('main', class_='nhsuk-main-wrapper')
        
        if main_content:
            job_posting.job_summary = clean_list(extract_tag_text(main_content, 'h3', 'summary'))
            job_posting.main_duties = clean_list(extract_tag_text(main_content, 'h3', 'duties'))
            job_posting.team_structure = clean_list(extract_tag_text(main_content, 'h3', 'about us'))
            job_posting.qualifications = clean_list(extract_qualifications(main_content, 'h2', 'specification'))
            job_posting.job_description = clean_list(extract_tag_text(main_content, 'h2', 'description'))
    
            job_posting.working_pattern = extract_tag_text(main_content, 'h3', 'working pattern')
            employer_name_tag = main_content.find('p', id='employer_name_details')
            job_posting.employer_name = clean_text(employer_name_tag.text) if employer_name_tag else 'N/A'
    
            address_fields = ['employer_address_line_1_a', 'employer_address_line_2_b', 'employer_town_c', 'employer_postcode_e']
            job_posting.employer_address = clean_text(' '.join([clean_text(main_content.find('p', id=field).text or '')
                                                                for field in address_fields if main_content.find('p', id=field)])) or 'N/A'
    
            employer_contact_tag = main_content.find('p', id='employer_website_url')
            if employer_contact_tag:
                employer_contact_link = employer_contact_tag.find('a', id='employer_website_url_link')
                job_posting.employer_contact = clean_text(employer_contact_link['href']) if employer_contact_link else 'N/A'
            else:
                job_posting.employer_contact = 'N/A'
    
            job_posting.disclosure_check = clean_text(main_content.find('div', id='dbs-container').text if main_content.find('div', id='dbs-container') else 'N/A')
            job_posting.certificate_of_sponsorship = clean_text(main_content.find('h3', id='tier-two-sponsorship').find_next('p').text if main_content.find('h3', id='tier-two-sponsorship') else 'N/A')
            job_posting.uk_registration = clean_text(main_content.find('h3', id='uk-registration').find_next('p').text if main_content.find('h3', id='uk-registration') else 'N/A')
            job_posting.pay_scheme = clean_text(main_content.find('p', id='payscheme-type').text if main_content.find('p', id='payscheme-type') else 'None')
            job_posting.grade = clean_text(main_content.find('p', id='payscheme-band').text if main_content.find('p', id='payscheme-band') else 'None')
            job_posting.reference_number = clean_text(main_content.find('p', id='trac-job-reference').text if main_content.find('p', id='trac-job-reference') else 'None')
            job_posting.duration = clean_text(main_content.find('p', id='contract_duration').text if main_content.find('p', id='contract_duration') else 'None')

    except Exception as e:
        print(f"Error extracting details for {job_posting.title}: {e}")
        
    
async def scrape_jobs_playwright(url):
    job_listings = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        page_number = 1
        
        try:
            while True:
                await page.goto(url)
                await page.wait_for_selector('.nhsuk-list.search-results')
                soup = BeautifulSoup(await page.content(), 'html.parser')
                job_elements = soup.find_all('li', class_='nhsuk-list-panel')

                for job in job_elements:
                    title_tag = job.find('a', {'data-test': 'search-result-job-title'})
                    title = clean_text(title_tag.text) if title_tag else 'N/A'
                    job_link = f"https://www.jobs.nhs.uk{title_tag['href']}" if title_tag else 'N/A'
                    location = clean_text(job.find('div', {'data-test': 'search-result-location'}).text or 'N/A')
                    salary = clean_text(job.find('li', {'data-test': 'search-result-salary'}).find('strong').text or 'N/A')
                    date_posted = clean_text(job.find('li', {'data-test': 'search-result-publicationDate'}).find('strong').text or 'N/A')
                    closing_date = clean_text(job.find('li', {'data-test': 'search-result-closingDate'}).find('strong').text or 'N/A')
                    contract_type = clean_text(job.find('li', {'data-test': 'search-result-jobType'}).find('strong').text or 'N/A')
                    working_pattern = clean_text(job.find('li', {'data-test': 'search-result-workingPattern'}).find('strong').text or 'N/A')

                    job_posting = JobPosting(title, location, salary, date_posted, closing_date, contract_type, working_pattern, job_link)
                    await extract_job_details(page, job_posting)
                    job_listings.append(job_posting)

                print(f"Page {page_number} scraped")

                next_page_tag = soup.find('li', class_='nhsuk-pagination-item--next')
                if next_page_tag:
                    next_page_link = next_page_tag.find('a', {'data-test': 'search-next-page'})
                    if next_page_link and 'href' in next_page_link.attrs:
                        url = f"https://www.jobs.nhs.uk{next_page_link['href']}"
                        page_number += 1
                    else:
                        break
                else:
                    break

        except Exception as e:
            print(f"Error while scraping page {page_number}: {e}")
        finally:
            await browser.close()

    return job_listings


async def main():
    base_url = "https://www.jobs.nhs.uk/candidate/search/results?"
    keyword = "FY2, CT1, CT2, ST1, ST2, ST3, LAS, Trust doctor, Trust grade"
    pay_band = "SPECIALTY_DOCTOR,FOUNDATION_DOCTOR,DOCTOR_OTHER"
    pay_range = "30-40,40-50"
    sort_by = "publicationDateDesc"
    language = "en"

    url = (f"{base_url}keyword={keyword.replace(' ', '%20')}&payBand={pay_band}&payRange={pay_range}"
           f"&skipPhraseSuggester=true&searchFormType=sortBy&sort={sort_by}&language={language}")

    start_time = time.time()

    # Directly await the scraping function without running asyncio.run()
    jobs = await scrape_jobs_playwright(url)

    print(f"Scraping completed in {time.time() - start_time:.2f} seconds")

    for job in jobs:
        print(job)
        print("-" * 50)

# To execute the code, just call the main() function inside an async loop in Jupyter
await main()


Page 1 scraped
Page 2 scraped
Page 3 scraped


CancelledError: 